# 7. Afhankelijkheid tussen waterstand en wind

Python-versie van de productie-chunk van het R-script **`dependence-wlwind10-impr.Rmd`**
(*Produce s-values and corrcoefs for all tidal stations from SEAS5-DCSM-5 data*).

Per station en windrichtingssector worden uit de SEAS5-DCSM5-data twee maten geschat voor de afhankelijkheid
tussen extreme hoogwaterstand en windschuifspanning:
- de **s-waarde**: de standaardafwijking van het verschil tussen waterstand en windschuifspanning, beide
  getransformeerd naar een standaard-exponentiële verdeling (zie paragraaf 4.5 van het rapport
  WBI-2017 basisstochasten);
- de **extremale correlatiecoëfficiënt** van een Gaussisch copulamodel.

Beide gebruiken de drempel `pthreshold[kthresh]` (= 0,012). In dit notebook worden geen toevalsgetallen gebruikt.
De eerdere chunks van het R-script vergelijken metingen en modeldata alleen in figuren; die zijn niet overgenomen.

**Gebruik:** controleer de instellingen hieronder en voer daarna alle cellen uit. Dit notebook heeft alleen de
invoerdata nodig. De waarden per station hangen niet af van de andere stations: je kunt elke selectie van stations
in elke volgorde doorrekenen. De csv-bestanden hebben één regel per station, in de volgorde van `stations`; alleen
met de volledige lijst in de standaardvolgorde zijn ze tekstueel gelijk aan die van R.

Uitvoer (dezelfde bestanden als het R-script, in `output_path`):
- `svalues_2035_{22hdeg,30deg}_0p012_versionFeb2026.csv`, `corrcoefs_2035_{22hdeg,30deg}_0p012_versionFeb2026.csv`
- dezelfde tabellen als `svalues_2035_*.pkl` / `corrcoefs_2035_*.pkl`

## Instellingen
De standaardwaarden van de mappen staan in `kuststatistiek/settings.py`.

In [ ]:
from kuststatistiek import settings

input_path = settings.DATA_PATH  # map met de aangeleverde invoerdata
output_path = settings.OUTPUT_PATH  # map voor de resultaten

stations = [
    "hansweert",
    "vlissing",
    "os11",
    "hoekvanh",
    "ijmuiden",
    "denhelde",
    "denoever",
    "westters",
    "harlinge",
    "lauwerso",
    "huibertg",
    "delfzijl",
]

## Voorbereiding

In [ ]:
import numpy as np

from kuststatistiek.settings import Paths, Bins, ProductionBins, wl_settings
from kuststatistiek.io.read_wind import read_seas5_multi
from kuststatistiek.gw import assigncat, rank_transform
from kuststatistiek.rutils import (
    r_quantile,
    r_sd,
    r_mean,
    r_as_character,
    r_signif,
    write_r_csv,
    save_pkl,
)

paths = Paths(input_path=input_path, output_path=output_path)
WL = wl_settings()
bins, pbins = (
    Bins(),
    ProductionBins(),
)  # sectoren van 22,5 graden (bins) en van 30 graden (pbins)
p = WL.pthreshold[
    WL.kthresh - 1
]  # drempel: het deel van de data dat wordt gebruikt (0,012)

# Net als in R gebruikt Hansweert de SEAS5-DCSM5-data van Vlissingen.
stationref = ["vlissing" if station == "hansweert" else station for station in stations]

## De twee schatters (R-functies `estimates.R` en `estimaterho.R`)

In [ ]:
def estimates(wl, windsp, winddir, lbin0, ubin0, n):
    """s-waarde per sector: standaardafwijking van het verschil van de exponentieel getransformeerde variabelen,
    voor het deel n van de data met de grootste som."""
    cat0 = assigncat(winddir, lbin0, ubin0)
    svalue = np.full(len(lbin0), np.nan)
    for i in range(len(lbin0)):
        ind = cat0[:, i]
        m = int(ind.sum())
        wlt = rank_transform(wl[ind], m)
        spt = rank_transform(windsp[ind], m)
        s = wlt + spt
        t = -wlt + spt
        prob = 1 - n / len(s) if n > 1 else 1 - n
        qs = r_quantile(s, prob)[0]
        svalue[i] = r_sd(t[s > qs])
    return svalue


def estimaterho(wl, windsp, winddir, lbin0, ubin0, n):
    """Extremale correlatiecoëfficiënt per sector.
    Let op (zoals in R): n wordt in de eerste sector vervangen door een aantal, dat ook voor alle volgende
    sectoren wordt gebruikt."""
    cat0 = assigncat(winddir, lbin0, ubin0)
    rho = np.full(len(lbin0), np.nan)
    for i in range(len(lbin0)):
        ind = cat0[:, i]
        m = int(ind.sum())
        s = np.minimum(rank_transform(wl[ind], m), rank_transform(windsp[ind], m))
        ls = len(s)
        s = -np.sort(-s)
        if n < 1:
            n = int(np.round(n * ls))
        sigma = r_mean(s[:n]) - s[n]
        rho[i] = sigma * 2 - 1
    return rho

## SEAS5-DCSM5-data inlezen
Waterstand `wl`, windschuifspanning `mstress` en richting van de windschuifspanning `dstress` per station.

In [ ]:
data = {}
for station in dict.fromkeys(stationref):  # elk station één keer
    d = read_seas5_multi(station, paths.input_path)
    data[station] = (d["wl"], d["mstress"], d["dstress"])

## s-waarden en correlatiecoëfficiënten (R-chunk `svalues`)
Eerst voor sectoren van 22,5 graden, daarna voor sectoren van 30 graden.

In [ ]:
outchar = r_as_character(r_signif(p, 2)).replace(
    ".", "p"
)  # "0p012" in de bestandsnamen
for tag, lb, ub, cats in [
    ("22hdeg", bins.lbin0, bins.ubin0, bins.cats0),
    ("30deg", pbins.lbin30, pbins.ubin30, pbins.cats30),
]:
    TT = np.round(
        np.array([estimates(*data[s], lb, ub, p) for s in stationref]), 1
    )  # s-waarden
    UU = np.round(
        np.array([estimaterho(*data[s], lb, ub, p) for s in stationref]), 2
    )  # correlatiecoëfficiënten
    colnames = ["stationsys5"] + [
        "X" + r_as_character(c) for c in cats
    ]  # kolomnamen zoals R's data.frame()
    directions = [r_as_character(c) for c in cats]
    save_pkl(
        paths.out(f"svalues_2035_{tag}.pkl"),
        TT=TT,
        station=list(stations),
        direction=directions,
        p=p,
    )
    save_pkl(
        paths.out(f"corrcoefs_2035_{tag}.pkl"),
        UU=UU,
        station=list(stations),
        direction=directions,
        p=p,
    )
    write_r_csv(
        paths.out(f"svalues_2035_{tag}_{outchar}_versionFeb2026.csv"),
        [list(stations)] + [TT[:, j] for j in range(TT.shape[1])],
        colnames,
        row_names=False,
    )
    write_r_csv(
        paths.out(f"corrcoefs_2035_{tag}_{outchar}_versionFeb2026.csv"),
        [list(stations)] + [UU[:, j] for j in range(UU.shape[1])],
        colnames,
        row_names=False,
    )
    print(f"afhankelijkheid {tag} klaar")